# 02_graphgrued_build_weekly_active_dataset

This notebook **changes almost the entire data construction approach** to align more closely with GraphGRU-Ed:

* Keeps only **active learners** (with at least 1 interaction), similar to the paper;
* Aggregates behavior over the **first 4 weeks after `enroll_time`**;
* Builds **weekly features** instead of a static snapshot;
* Retains **missingness** in the weekly features for graph-based imputation in the subsequent step;
* Creates both a **binary completion label** (to maintain comparability with the old pipeline) and a **5-level surrogate label** that is closer to the paper's concept.

> Important note:
>
> The GraphGRU-Ed paper normalizes scores from various grading systems into 5 levels: A/B/C/D/E. Since your current data and notebook do not have an independent and clear final-grade table like the paper, this notebook creates:
>
> 1. `is_completed` based on your previous logic;
> 2. `risk_label_5_surrogate` based on normalized course performance / completion proxy.
>
> If you have a more standardized final course grade table later, you simply need to replace the `build_surrogate_5level_label` section.


In [11]:

# =========================
# 0) CONFIG
# =========================
RAW_ROOT = "Files/temp"
USE_RAW_JSON = True

OUTPUT_TABLE = "dbo.tbl_graphgrued_weekly_active_base"
ACTIVE_USER_TABLE = "dbo.tbl_graphgrued_active_users"

COMPLETION_THRESHOLD = 0.60
SCORE_PCT_THRESHOLD = 0.01
N_WEEKS = 4
DAYS_PER_WEEK = 7
MAX_DAYS = N_WEEKS * DAYS_PER_WEEK


StatementMeta(, b736590c-1d4c-4ea5-bc3c-b6ba0c1178b5, 13, Finished, Available, Finished, False)

In [12]:

# =========================
# 1) IMPORTS
# =========================
import re
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StringType, TimestampType, DoubleType


StatementMeta(, b736590c-1d4c-4ea5-bc3c-b6ba0c1178b5, 14, Finished, Available, Finished, False)

In [13]:

# =========================
# 2) HELPERS
# =========================
def normalize_colnames(df, default_prefix="No_name"):
    seen = set()
    new_names = []
    for i, c in enumerate(df.columns):
        name = "" if c is None else str(c).strip()
        if name == "":
            name = f"{default_prefix}_{i}"
        name = name.replace("`", "")
        name = re.sub(r"[^0-9a-zA-Z_]", "_", name)
        name = re.sub(r"_+", "_", name).strip("_")
        if name == "":
            name = f"{default_prefix}_{i}"
        base = name
        k = 1
        while name in seen:
            name = f"{base}_{k}"
            k += 1
        seen.add(name)
        new_names.append(name)
    return df.toDF(*new_names)


def ensure_prefixed(df, col_name, prefix):
    return df.withColumn(
        col_name,
        F.when(
            F.col(col_name).cast("string").startswith(prefix),
            F.col(col_name).cast("string")
        ).otherwise(F.concat(F.lit(prefix), F.col(col_name).cast("string")))
    )


def safe_divide(num_col, den_col):
    return F.when(den_col.isNull() | (den_col == 0), F.lit(0.0)).otherwise(num_col / den_col)


def coalesce_timestamp_expr(df, candidates, unit="auto"):
    exprs = []
    for c in candidates:
        if c in df.columns:
            col = F.col(c)
            if unit == "ms":
                exprs.append((col / 1000).cast("timestamp"))
            elif unit == "s":
                exprs.append(col.cast("timestamp"))
            else:
                exprs.append(
                    F.when(col.cast("double") > F.lit(10_000_000_000), (col / 1000).cast("timestamp"))
                     .otherwise(col.cast("timestamp"))
                )
    if not exprs:
        return None
    out = exprs[0]
    for e in exprs[1:]:
        out = F.coalesce(out, e)
    return out


def add_week_index(df, ts_col, enroll_col, n_weeks=4):
    return (
        df.withColumn("event_day", F.to_date(ts_col))
          .withColumn("enroll_day", F.to_date(enroll_col))
          .withColumn("day_offset", F.datediff(F.col("event_day"), F.col("enroll_day")))
          .withColumn(
              "week_idx",
              F.when((F.col("day_offset") >= 0) & (F.col("day_offset") < n_weeks * 7), F.floor(F.col("day_offset") / 7) + 1)
          )
          .filter(F.col("week_idx").isNotNull())
    )


def pivot_weekly(df, index_cols, value_col, prefix, agg="sum", fill_none=False):
    if agg == "sum":
        aggr = F.sum(value_col)
    elif agg == "count":
        aggr = F.count(value_col)
    elif agg == "count_distinct":
        aggr = F.countDistinct(value_col)
    elif agg == "avg":
        aggr = F.avg(value_col)
    else:
        raise ValueError(f"Unsupported agg={agg}")

    out = (
        df.groupBy(*index_cols)
          .pivot("week_idx", list(range(1, N_WEEKS + 1)))
          .agg(aggr)
    )
    for w in range(1, N_WEEKS + 1):
        old = str(w)
        new = f"{prefix}_week{w}"
        if old in out.columns:
            out = out.withColumnRenamed(old, new)
        else:
            out = out.withColumn(new, F.lit(None).cast("double"))
    if fill_none:
        out = out.fillna(0.0, subset=[f"{prefix}_week{w}" for w in range(1, N_WEEKS + 1)])
    return out


def build_surrogate_5level_label(df, perf_col="course_norm_score", completion_col="completion_rate"):
    # Gần tinh thần paper nhưng KHÔNG thay thế được grade normalization thật nếu bạn có bảng điểm chuẩn.
    return (
        df.withColumn(
            "risk_label_5_surrogate",
            F.when((F.col(completion_col) <= 0.0) | F.col(perf_col).isNull(), F.lit("E"))
             .when((F.col(perf_col) >= 0.90) & (F.col(completion_col) >= 0.90), F.lit("A"))
             .when((F.col(perf_col) >= 0.80) & (F.col(completion_col) >= 0.80), F.lit("B"))
             .when((F.col(perf_col) >= 0.60) & (F.col(completion_col) >= 0.60), F.lit("C"))
             .otherwise(F.lit("D"))
        )
    )


StatementMeta(, b736590c-1d4c-4ea5-bc3c-b6ba0c1178b5, 15, Finished, Available, Finished, False)

In [14]:

# =========================
# 3) LOAD DATA
# =========================
if USE_RAW_JSON:
    course_raw = spark.read.json(f"{RAW_ROOT}/course.json")
    user_raw = spark.read.json(f"{RAW_ROOT}/user.json")
    comment_raw = spark.read.json(f"{RAW_ROOT}/comment.json")
    reply_raw = spark.read.json(f"{RAW_ROOT}/reply.json")
    user_video_raw = spark.read.json(f"{RAW_ROOT}/user-video.json")
    user_problem_raw = spark.read.json(f"{RAW_ROOT}/user-problem.json")
    problem_raw = spark.read.json(f"{RAW_ROOT}/problem.json")
    exercise_problem_raw = spark.read.csv(
        f"{RAW_ROOT}/exercise-problem.txt",
        sep="\t",
        schema="exercise_id STRING, problem_id STRING"
    )
else:
    raise ValueError("Notebook này hiện đang viết theo raw JSON MOOCCubeX.")

course_raw = normalize_colnames(course_raw)
user_raw = normalize_colnames(user_raw)
comment_raw = normalize_colnames(comment_raw)
reply_raw = normalize_colnames(reply_raw)
user_video_raw = normalize_colnames(user_video_raw)
user_problem_raw = normalize_colnames(user_problem_raw)
problem_raw = normalize_colnames(problem_raw)
exercise_problem_raw = normalize_colnames(exercise_problem_raw)

print("Loaded raw JSONs successfully")


StatementMeta(, b736590c-1d4c-4ea5-bc3c-b6ba0c1178b5, 16, Finished, Available, Finished, False)

Loaded raw JSONs successfully


In [15]:
exercise_problem_raw = spark.read.csv(
        f"{RAW_ROOT}/exercise-problem.txt",
        sep="\t",
        schema="exercise_id STRING, problem_id STRING"
    )

StatementMeta(, b736590c-1d4c-4ea5-bc3c-b6ba0c1178b5, 17, Finished, Available, Finished, False)

In [16]:
# 1. Explode mảng bên ngoài để tách mỗi video_id thành 1 dòng
# Cột 'seq' hiện tại là mảng các struct chứa video_id và segment
df_exploded_video = user_video_raw.select(
    "user_id", 
    F.explode("seq").alias("video_data")
)

# 2. Explode tiếp mảng 'segment' bên trong video_data
# Lúc này ta truy cập vào video_data.segment để lấy mảng các đoạn cắt
df_exploded_segments = df_exploded_video.select(
    "user_id",
    F.col("video_data.video_id").alias("video_id"),
    F.explode("video_data.segment").alias("seg")
)

# 3. Làm phẳng (flatten) các trường trong struct 'seg' ra thành các cột riêng biệt
user_video_raw = df_exploded_segments.select(
    "user_id",
    "video_id",
    F.col("seg.start_point").alias("start_point"),
    F.col("seg.end_point").alias("end_point"),
    F.col("seg.speed").alias("speed"),
    F.col("seg.local_start_time").alias("local_start_time")
)

StatementMeta(, b736590c-1d4c-4ea5-bc3c-b6ba0c1178b5, 18, Finished, Available, Finished, False)

In [17]:
display(user_video_raw.filter(F.col('end_point') < F.col('start_point')).limit(10))

StatementMeta(, b736590c-1d4c-4ea5-bc3c-b6ba0c1178b5, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bd66a423-0cfb-4f08-bab0-a9024ab2af34)

In [18]:

# =========================
# 4) PREPARE CORE ENTITY TABLES
# =========================
course_df = (
    course_raw
    .select(
        F.col("id").alias("course_id"),
        F.col("name").alias("course_name"),
        F.col("about"),
        F.col("prerequisites")
    )
    .dropDuplicates(["course_id"])
)
course_df = ensure_prefixed(course_df, "course_id", "C_")

course_field_df = (
    course_raw
    .select(F.col("id").alias("course_id"), F.explode_outer(F.col("field")).alias("field_id"))
    .dropDuplicates(["course_id", "field_id"])
)
course_field_df = ensure_prefixed(course_field_df, "course_id", "C_")

course_resource_df = (
    course_raw
    .select(F.col("id").alias("course_id"), F.explode_outer(F.col("resource")).alias("resource"))
    .select(
        F.col("course_id"),
        F.col("resource.resource_id").alias("resource_id"),
        F.col("resource.chapter").alias("chapter"),
        F.explode_outer(F.col("resource.titles")).alias("title")
    )
    .dropDuplicates(["course_id", "resource_id", "title"])
)
course_resource_df = ensure_prefixed(course_resource_df, "course_id", "C_")

user_df = (
    user_raw
    .select(
        F.col("id").alias("user_id"),
        F.col("gender"),
        F.col("year_of_birth"),
        F.col("school").alias("school_id")
    )
    .dropDuplicates(["user_id"])
)

user_course_enroll_df = (
    user_raw
    .select(F.col("id").alias("user_id"), F.posexplode_outer(F.col("course_order")).alias("idx", "course_index"))
    .join(
        user_raw.select(F.col("id").alias("uid"), F.col("enroll_time")),
        F.col("user_id") == F.col("uid"),
        "left"
    )
    .select(
        F.col("user_id"),
        F.col("course_index"),
        F.col("enroll_time")[F.col("idx")].alias("enroll_time")
    )
    .dropDuplicates(["user_id", "course_index"])
)
user_course_enroll_df = ensure_prefixed(user_course_enroll_df, "course_index", "C_")
user_course_enroll_df = user_course_enroll_df.withColumn(
    "enroll_time_ts",
    F.when(F.col("enroll_time").cast("double") > 10_000_000_000, (F.col("enroll_time") / 1000).cast("timestamp"))
     .otherwise(F.col("enroll_time").cast("timestamp"))
)

comment_df = comment_raw
reply_df = reply_raw

user_video_df = (
    user_video_raw
    .select(F.col("user_id"), F.col("video_id"))
    .dropDuplicates(["user_id", "video_id"])
)
user_video_df = ensure_prefixed(user_video_df, "video_id", "V_")

user_problem_df = (
    user_problem_raw
    .select("user_id", "problem_id", "is_correct", F.col("score").cast("double").alias("score"))
    .dropDuplicates(["user_id", "problem_id"])
)
user_problem_df = ensure_prefixed(user_problem_df, "problem_id", "Pm_")

problem_df = problem_raw.select(
        F.concat(F.lit("Pm_"), F.col("problem_id")).alias("problem_id"),
        *[c for c in problem_raw.columns if c != "problem_id"]
    )
problem_df = ensure_prefixed(problem_df, "problem_id", "Pm_")

exercise_problem_df = exercise_problem_raw.select("exercise_id", "problem_id").dropDuplicates(["exercise_id", "problem_id"])
exercise_problem_df = ensure_prefixed(exercise_problem_df, "exercise_id", "Ex_")
exercise_problem_df = ensure_prefixed(exercise_problem_df, "problem_id", "Pm_")

course_video_map = (
    course_resource_df.filter(F.col("resource_id").startswith("V_"))
    .select("course_id", F.col("resource_id").alias("video_id"), F.col("chapter").cast("double").alias("video_chapter"))
    .dropDuplicates(["course_id", "video_id"])
)

course_exercise_df = (
    course_resource_df.filter(F.col("resource_id").startswith("Ex_"))
    .select("course_id", F.col("resource_id").alias("exercise_id"))
    .dropDuplicates(["course_id", "exercise_id"])
)

course_problem_map = (
    course_exercise_df.join(exercise_problem_df, on="exercise_id", how="inner")
    .select("course_id", "problem_id")
    .dropDuplicates(["course_id", "problem_id"])
)


StatementMeta(, b736590c-1d4c-4ea5-bc3c-b6ba0c1178b5, 20, Finished, Available, Finished, False)

In [19]:
# =========================
# 5) DEFINE ACTIVE LEARNER COHORT (FIXED: course-level active)
# =========================
video_ts_expr = coalesce_timestamp_expr(
    user_video_raw,
    candidates=["local_start_time", "timestamp", "created_time", "event_time"],
    unit="auto"
)

video_active_uc = (
    user_video_raw
    .withColumn(
        "video_id",
        F.when(F.col("video_id").cast("string").startswith("V_"), F.col("video_id").cast("string"))
         .otherwise(F.concat(F.lit("V_"), F.col("video_id").cast("string")))
    )
    .select("user_id", "video_id", video_ts_expr.alias("ts"))
    .dropna(subset=["ts"])
    .join(course_video_map.select("video_id", "course_id").dropDuplicates(), on="video_id", how="inner")
    .select("user_id", "course_id", "ts")
)

problem_ts_expr = coalesce_timestamp_expr(
    user_problem_raw,
    candidates=["submit_time", "local_submit_time", "timestamp", "created_time", "event_time"],
    unit="auto"
)

problem_active_uc = (
    user_problem_raw
    .withColumn(
        "problem_id",
        F.when(F.col("problem_id").cast("string").startswith("Pm_"), F.col("problem_id").cast("string"))
         .otherwise(F.concat(F.lit("Pm_"), F.col("problem_id").cast("string")))
    )
    .select("user_id", "problem_id", problem_ts_expr.alias("ts"))
    .dropna(subset=["ts"])
    .join(course_problem_map.select("problem_id", "course_id").dropDuplicates(), on="problem_id", how="inner")
    .select("user_id", "course_id", "ts")
)

active_parts = [video_active_uc, problem_active_uc]

comment_ts_expr = coalesce_timestamp_expr(comment_raw, ["timestamp", "created_time", "post_time", "comment_time", "date", "time"], unit="auto")
if comment_ts_expr is not None and "resource_id" in comment_raw.columns and "user_id" in comment_raw.columns:
    comment_active_uc = (
        comment_raw
        .select("user_id", F.col("resource_id").cast("string").alias("resource_id"), comment_ts_expr.alias("ts"))
        .dropna(subset=["ts"])
        .join(course_resource_df.select("course_id", "resource_id").dropDuplicates(), on="resource_id", how="inner")
        .select("user_id", "course_id", "ts")
    )
    active_parts.append(comment_active_uc)

reply_ts_expr = coalesce_timestamp_expr(reply_raw, ["timestamp", "created_time", "reply_time", "date", "time"], unit="auto")
if reply_ts_expr is not None and "resource_id" in reply_raw.columns and "user_id" in reply_raw.columns:
    reply_active_uc = (
        reply_raw
        .select("user_id", F.col("resource_id").cast("string").alias("resource_id"), reply_ts_expr.alias("ts"))
        .dropna(subset=["ts"])
        .join(course_resource_df.select("course_id", "resource_id").dropDuplicates(), on="resource_id", how="inner")
        .select("user_id", "course_id", "ts")
    )
    active_parts.append(reply_active_uc)

activity_uc_df = active_parts[0]
for part in active_parts[1:]:
    activity_uc_df = activity_uc_df.unionByName(part, allowMissingColumns=True)

activity_uc_df = activity_uc_df.dropDuplicates(["user_id", "course_id", "ts"])

active_enroll_df = (
    user_course_enroll_df
    .filter(F.col("enroll_time_ts").isNotNull())
    .select("user_id", F.col("course_index").alias("course_id"), "enroll_time_ts")
    .dropDuplicates(["user_id", "course_id"])
    .join(activity_uc_df, on=["user_id", "course_id"], how="inner")
    .withColumn("days_from_enroll", F.datediff(F.to_date("ts"), F.to_date("enroll_time_ts")))
    .filter((F.col("days_from_enroll") >= 0) & (F.col("days_from_enroll") < MAX_DAYS))
    .select("user_id", "course_id", "enroll_time_ts")
    .dropDuplicates(["user_id", "course_id"])
)

active_users_df = active_enroll_df.select("user_id").dropDuplicates()

print("Active users   :", active_users_df.count())
print("Active enrolls :", active_enroll_df.count())

StatementMeta(, b736590c-1d4c-4ea5-bc3c-b6ba0c1178b5, 21, Finished, Available, Finished, False)

Active users   : 95129
Active enrolls : 100409


In [20]:
# =========================
# 6) VIDEO WEEKLY FEATURES (FIXED)
# =========================
video_ts_expr = coalesce_timestamp_expr(
    user_video_raw,
    candidates=["local_start_time", "timestamp", "created_time", "event_time"],
    unit="auto"
)

video_seg_df = (
    user_video_raw
    .withColumn(
        "video_id",
        F.when(F.col("video_id").cast("string").startswith("V_"), F.col("video_id").cast("string"))
         .otherwise(F.concat(F.lit("V_"), F.col("video_id").cast("string")))
    )
    .withColumn("delta_point", (F.col("end_point").cast("double") - F.col("start_point").cast("double")))
    .withColumn("forward_progress", F.when(F.col("delta_point") > 0, F.col("delta_point")).otherwise(F.lit(0.0)))
    .withColumn("rewind_distance", F.when(F.col("delta_point") < 0, -F.col("delta_point")).otherwise(F.lit(0.0)))
    .withColumn("rewind_flag", F.when(F.col("delta_point") < 0, 1).otherwise(0))
    .select(
        "user_id",
        "video_id",
        video_ts_expr.alias("ts"),
        "forward_progress",
        "rewind_distance",
        "rewind_flag"
    )
    .dropna(subset=["ts"])
)

video_event_df = (
    video_seg_df
    .join(course_video_map, on="video_id", how="inner")
    .join(active_enroll_df, on=["user_id", "course_id"], how="inner")
)

video_event_df = add_week_index(
    video_event_df,
    ts_col=F.col("ts"),
    enroll_col=F.col("enroll_time_ts"),
    n_weeks=N_WEEKS
)

video_time_weekly_df = pivot_weekly(
    video_event_df,
    ["user_id", "course_id"],
    "forward_progress",
    "video_watch_time",
    agg="sum",
    fill_none=False
)

video_count_weekly_df = pivot_weekly(
    video_event_df,
    ["user_id", "course_id"],
    "video_id",
    "video_count",
    agg="count_distinct",
    fill_none=False
)

video_rewind_count_weekly_df = pivot_weekly(
    video_event_df,
    ["user_id", "course_id"],
    "rewind_flag",
    "rewind_count",
    agg="sum",
    fill_none=False
)

video_rewind_dist_weekly_df = pivot_weekly(
    video_event_df,
    ["user_id", "course_id"],
    "rewind_distance",
    "rewind_distance",
    agg="sum",
    fill_none=False
)

video_chapter_weekly_df = pivot_weekly(
    video_event_df,
    ["user_id", "course_id"],
    "video_chapter",
    "video_avg_chapter",
    agg="avg",
    fill_none=False
)

StatementMeta(, b736590c-1d4c-4ea5-bc3c-b6ba0c1178b5, 22, Finished, Available, Finished, False)

In [21]:

# =========================
# 7) PROBLEM WEEKLY FEATURES
# =========================
problem_ts_expr = coalesce_timestamp_expr(
    user_problem_raw,
    candidates=["submit_time", "local_submit_time", "timestamp", "created_time", "event_time"],
    unit="auto"
)

if problem_ts_expr is None:
    print("No explicit problem timestamp found. Weekly problem features will be empty / null.")
    problem_weekly_df = active_enroll_df.select("user_id", "course_id")
else:
    up_event_df = (
        user_problem_raw
        .select(
            F.col("user_id"),
            F.col("problem_id"),
            F.col("is_correct").cast("int").alias("is_correct_i"),
            F.col("score").cast("double").alias("score_d"),
            problem_ts_expr.alias("ts")
        )
        .withColumn("problem_id", F.when(F.col("problem_id").cast("string").startswith("Pm_"), F.col("problem_id").cast("string")).otherwise(F.concat(F.lit("Pm_"), F.col("problem_id").cast("string"))))
        .dropna(subset=["ts"])
        .join(course_problem_map, on="problem_id", how="inner")
        .join(active_enroll_df, on=["user_id", "course_id"], how="inner")
    )
    up_event_df = add_week_index(up_event_df, ts_col=F.col("ts"), enroll_col=F.col("enroll_time_ts"), n_weeks=N_WEEKS)

    if "max_score" in problem_df.columns:
        max_score_df = problem_df.select("problem_id", F.col("max_score").cast("double").alias("max_score"))
    elif "score" in problem_df.columns:
        max_score_df = problem_df.select("problem_id", F.col("score").cast("double").alias("max_score"))
    else:
        max_score_df = problem_df.select("problem_id").withColumn("max_score", F.lit(None).cast("double"))

    up_event_df = (
        up_event_df
        .join(max_score_df, on="problem_id", how="left")
        .withColumn("normalized_score", safe_divide(F.coalesce(F.col("score_d"), F.lit(0.0)), F.col("max_score")))
        .withColumn(
            "is_passed",
            F.when((F.col("is_correct_i") == 1) | (F.col("normalized_score") >= F.lit(SCORE_PCT_THRESHOLD)), F.lit(1)).otherwise(F.lit(0))
        )
    )

    problem_attempt_weekly_df = pivot_weekly(up_event_df, ["user_id", "course_id"], "problem_id", "problem_attempt_count", agg="count_distinct", fill_none=False)
    problem_score_weekly_df = pivot_weekly(up_event_df, ["user_id", "course_id"], "score_d", "total_score", agg="sum", fill_none=False)
    problem_correct_weekly_df = pivot_weekly(up_event_df, ["user_id", "course_id"], "is_correct_i", "correct_answers", agg="sum", fill_none=False)
    problem_pass_weekly_df = pivot_weekly(up_event_df, ["user_id", "course_id"], "is_passed", "completed_questions", agg="sum", fill_none=False)

    problem_weekly_df = (
        active_enroll_df.select("user_id", "course_id")
        .join(problem_attempt_weekly_df, on=["user_id", "course_id"], how="left")
        .join(problem_score_weekly_df, on=["user_id", "course_id"], how="left")
        .join(problem_correct_weekly_df, on=["user_id", "course_id"], how="left")
        .join(problem_pass_weekly_df, on=["user_id", "course_id"], how="left")
    )


StatementMeta(, b736590c-1d4c-4ea5-bc3c-b6ba0c1178b5, 23, Finished, Available, Finished, False)

In [22]:

# =========================
# 8) COMMENT / REPLY WEEKLY FEATURES
# =========================
comment_ts_expr = coalesce_timestamp_expr(comment_raw, ["timestamp", "created_time", "post_time", "comment_time", "date", "time"], unit="auto")
reply_ts_expr = coalesce_timestamp_expr(reply_raw, ["timestamp", "created_time", "reply_time", "date", "time"], unit="auto")

comment_weekly_df = active_enroll_df.select("user_id", "course_id")
reply_weekly_df = active_enroll_df.select("user_id", "course_id")

if comment_ts_expr is not None and "resource_id" in comment_raw.columns and "user_id" in comment_raw.columns:
    comment_event_df = (
        comment_raw
        .select("user_id", "resource_id", comment_ts_expr.alias("ts"))
        .withColumn("resource_id", F.col("resource_id").cast("string"))
        .join(course_resource_df.select("course_id", "resource_id").dropDuplicates(["course_id", "resource_id"]), on="resource_id", how="inner")
        .join(active_enroll_df, on=["user_id", "course_id"], how="inner")
        .dropna(subset=["ts"])
    )
    comment_event_df = add_week_index(comment_event_df, ts_col=F.col("ts"), enroll_col=F.col("enroll_time_ts"), n_weeks=N_WEEKS)
    comment_weekly_df = pivot_weekly(comment_event_df, ["user_id", "course_id"], "resource_id", "comment_count", agg="count", fill_none=False)

if reply_ts_expr is not None and "user_id" in reply_raw.columns:
    # Reply data không phải lúc nào cũng có course/resource id rõ ràng, nên ưu tiên join theo user + course active nếu tìm được resource_id.
    if "resource_id" in reply_raw.columns:
        reply_event_df = (
            reply_raw
            .select("user_id", "resource_id", reply_ts_expr.alias("ts"))
            .withColumn("resource_id", F.col("resource_id").cast("string"))
            .join(course_resource_df.select("course_id", "resource_id").dropDuplicates(["course_id", "resource_id"]), on="resource_id", how="inner")
            .join(active_enroll_df, on=["user_id", "course_id"], how="inner")
            .dropna(subset=["ts"])
        )
        reply_event_df = add_week_index(reply_event_df, ts_col=F.col("ts"), enroll_col=F.col("enroll_time_ts"), n_weeks=N_WEEKS)
        reply_weekly_df = pivot_weekly(reply_event_df, ["user_id", "course_id"], "resource_id", "reply_count", agg="count", fill_none=False)


StatementMeta(, b736590c-1d4c-4ea5-bc3c-b6ba0c1178b5, 24, Finished, Available, Finished, False)

In [23]:

# =========================
# 9) STATIC COURSE / USER FEATURES
# =========================
course_static_df = (
    course_df
    .join(course_field_df.groupBy("course_id").agg(F.countDistinct("field_id").alias("num_fields")), on="course_id", how="left")
    .join(course_resource_df.groupBy("course_id").agg(F.countDistinct("resource_id").alias("num_resources")), on="course_id", how="left")
    .join(active_enroll_df.groupBy("course_id").agg(F.countDistinct("user_id").alias("num_active_enrollments")), on="course_id", how="left")
)

course_video_stats_df = (
    course_video_map.groupBy("course_id")
    .agg(
        F.countDistinct("video_id").alias("course_total_videos"),
        F.max("video_chapter").alias("course_max_chapter")
    )
)
course_static_df = course_static_df.join(course_video_stats_df, on="course_id", how="left")

user_static_df = (
    user_df
    .join(active_enroll_df.groupBy("user_id").agg(F.countDistinct("course_id").alias("num_courses_enrolled")), on="user_id", how="left")
)


StatementMeta(, b736590c-1d4c-4ea5-bc3c-b6ba0c1178b5, 25, Finished, Available, Finished, False)

In [24]:

# =========================
# 10) LABELS (binary + surrogate 5-level)
# =========================
if "max_score" in problem_df.columns:
    max_score_df = problem_df.select("problem_id", F.col("max_score").cast("double").alias("max_score"))
elif "score" in problem_df.columns:
    max_score_df = problem_df.select("problem_id", F.col("score").cast("double").alias("max_score"))
else:
    max_score_df = problem_df.select("problem_id").withColumn("max_score", F.lit(None).cast("double"))

completion_event_df = (
    user_problem_df
    .select("user_id", "problem_id", F.col("is_correct").cast("int").alias("is_correct_i"), F.col("score").cast("double").alias("score_d"))
    .join(max_score_df, on="problem_id", how="left")
    .withColumn("normalized_score", safe_divide(F.coalesce(F.col("score_d"), F.lit(0.0)), F.col("max_score")))
    .withColumn("is_passed", F.when((F.col("is_correct_i") == 1) | (F.col("normalized_score") >= F.lit(SCORE_PCT_THRESHOLD)), 1).otherwise(0))
    .join(course_problem_map, on="problem_id", how="inner")
)

course_problem_total_df = course_problem_map.groupBy("course_id").agg(F.countDistinct("problem_id").alias("total_problems"))

user_course_perf_df = (
    completion_event_df
    .groupBy("user_id", "course_id")
    .agg(
        F.sum("is_passed").alias("problems_passed"),
        F.avg("normalized_score").alias("course_norm_score"),
        F.sum("score_d").alias("course_raw_score_sum")
    )
)

label_df = (
    active_enroll_df.select("user_id", "course_id")
    .join(course_problem_total_df, on="course_id", how="left")
    .join(user_course_perf_df, on=["user_id", "course_id"], how="left")
    .fillna({"total_problems": 0, "problems_passed": 0, "course_norm_score": 0.0, "course_raw_score_sum": 0.0})
    .withColumn("completion_rate", safe_divide(F.col("problems_passed").cast("double"), F.col("total_problems").cast("double")))
    .withColumn("is_completed", F.when(F.col("completion_rate") >= F.lit(COMPLETION_THRESHOLD), F.lit(1)).otherwise(F.lit(0)))
)
label_df = build_surrogate_5level_label(label_df, perf_col="course_norm_score", completion_col="completion_rate")


StatementMeta(, b736590c-1d4c-4ea5-bc3c-b6ba0c1178b5, 26, Finished, Available, Finished, False)

In [26]:

# =========================
# 11) LIGHTWEIGHT PATH-AWARE / NORMALIZED FEATURES (small improvement vs paper)
# =========================
video_totals_df = (
    active_enroll_df.select("user_id", "course_id")
    .join(video_time_weekly_df, on=["user_id", "course_id"], how="left")
)
for w in range(1, N_WEEKS + 1):
    video_totals_df = video_totals_df.withColumn(f"video_watch_time_week{w}", F.col(f"video_watch_time_week{w}").cast("double"))

video_totals_df = video_totals_df.withColumn(
    "video_watch_time_total_4w",
    sum(F.coalesce(F.col(f"video_watch_time_week{w}"), F.lit(0.0)) for w in range(1, N_WEEKS + 1))
)

video_counts_df = active_enroll_df.select("user_id", "course_id").join(video_count_weekly_df, on=["user_id", "course_id"], how="left")
for w in range(1, N_WEEKS + 1):
    video_counts_df = video_counts_df.withColumn(f"video_count_week{w}", F.col(f"video_count_week{w}").cast("double"))
video_counts_df = video_counts_df.withColumn(
    "video_count_total_4w",
    sum(F.coalesce(F.col(f"video_count_week{w}"), F.lit(0.0)) for w in range(1, N_WEEKS + 1))
)

path_enrich_df = (
    active_enroll_df.select("user_id", "course_id")
    .join(video_counts_df.select("user_id", "course_id", "video_count_total_4w"), on=["user_id", "course_id"], how="left")
    .join(video_totals_df.select("user_id", "course_id", "video_watch_time_total_4w"), on=["user_id", "course_id"], how="left")
    .join(course_static_df.select("course_id", "course_total_videos", "course_max_chapter"), on="course_id", how="left")
    .join(video_chapter_weekly_df.select("user_id", "course_id", *[c for c in video_chapter_weekly_df.columns if c.startswith("video_avg_chapter_week")]), on=["user_id", "course_id"], how="left")
    .withColumn(
        "chapter_coverage_ratio_raw",
        safe_divide(F.col("video_count_total_4w"), F.col("course_total_videos"))
    )
    .withColumn(
        "chapter_coverage_ratio",
        F.when(F.col("chapter_coverage_ratio_raw") < 0, F.lit(0.0))
        .when(F.col("chapter_coverage_ratio_raw") > 1, F.lit(1.0))
        .otherwise(F.col("chapter_coverage_ratio_raw"))
    )
    .withColumn("avg_chapter_progress_4w", sum(F.coalesce(F.col(f"video_avg_chapter_week{w}"), F.lit(0.0)) for w in range(1, N_WEEKS + 1)) / F.lit(float(N_WEEKS)))
    .withColumn("chapter_progress_ratio_raw", safe_divide(F.col("avg_chapter_progress_4w"), F.col("course_max_chapter")))
    .withColumn(
        "chapter_progress_ratio",
        F.when(F.col("chapter_progress_ratio_raw") < 0, F.lit(0.0))
        .when(F.col("chapter_progress_ratio_raw") > 1, F.lit(1.0))
        .otherwise(F.col("chapter_progress_ratio_raw"))
    )
)


StatementMeta(, b736590c-1d4c-4ea5-bc3c-b6ba0c1178b5, 28, Finished, Available, Finished, False)

In [27]:

# =========================
# 12) BUILD FINAL WEEKLY BASE TABLE
# =========================
# =========================
# 12) BUILD FINAL WEEKLY BASE TABLE (Fixed version)
# =========================
model_df = active_enroll_df.select("user_id", "course_id", "enroll_time_ts")

data_parts = [
    user_static_df,
    course_static_df,
    video_time_weekly_df,
    video_count_weekly_df,
    video_rewind_count_weekly_df,
    video_rewind_dist_weekly_df,
    video_chapter_weekly_df,
    problem_weekly_df,
    comment_weekly_df,
    reply_weekly_df,
    label_df,
    path_enrich_df,
]

for part in data_parts:
    # 1. Xác định join keys
    join_keys = [c for c in ["user_id", "course_id"] if c in part.columns and c in model_df.columns]
    
    if not join_keys:
        continue # Bỏ qua nếu không có key chung
    
    # 2. XỬ LÝ LỖI AMBIGUOUS: 
    # Tìm các cột trùng tên (mà không phải là join keys) và xóa chúng khỏi bảng 'part'
    cols_to_drop = [c for c in part.columns if c in model_df.columns and c not in join_keys]
    part_clean = part.drop(*cols_to_drop)
    
    # 3. Thực hiện Join
    model_df = model_df.join(part_clean, on=join_keys, how="left")

# Xử lý các cột đặc thù
model_df = (
    model_df
    .withColumnRenamed("enroll_time_ts", "enroll_time")
    .withColumn("gender", F.col("gender").cast("string"))
    .withColumn("school_id", F.col("school_id").cast("string"))
)

# Xử lý Fillna cho các cột numeric quan trọng
fill_numeric_cols = [
    "num_fields", "num_resources", "num_active_enrollments", 
    "course_total_videos", "course_max_chapter", "num_courses_enrolled"
]

for c in fill_numeric_cols:
    if c in model_df.columns:
        # Sử dụng F.col(c) sau khi đã làm sạch cột trùng ở trên sẽ không còn lỗi Ambiguous
        model_df = model_df.withColumn(c, F.coalesce(F.col(c).cast("double"), F.lit(0.0)))

qc = model_df.select(
    F.count("*").alias("n_rows"),
    F.sum(F.when(F.col("video_watch_time_total_4w") < 0, 1).otherwise(0)).alias("neg_video_watch_time_total_4w"),
    F.sum(F.when(F.col("chapter_coverage_ratio") > 1, 1).otherwise(0)).alias("chapter_coverage_ratio_gt_1"),
    F.sum(F.when(F.col("course_name").isNull(), 1).otherwise(0)).alias("course_name_null"),
).collect()[0].asDict()

print(qc)

display(
    model_df.groupBy("risk_label_5_surrogate").count().orderBy("risk_label_5_surrogate")
)

print("Final rows:", model_df.count())
display(model_df.limit(20))

StatementMeta(, b736590c-1d4c-4ea5-bc3c-b6ba0c1178b5, 29, Finished, Available, Finished, False)

{'n_rows': 100409, 'neg_video_watch_time_total_4w': 0, 'chapter_coverage_ratio_gt_1': 0, 'course_name_null': 0}


SynapseWidget(Synapse.DataFrame, 27c93ec3-f988-4327-ad1d-caf370434d21)

Final rows: 100409


SynapseWidget(Synapse.DataFrame, 1571d5fa-dd3e-4c98-ac64-dd06e44cd4fa)

In [28]:

# =========================
# 13) SAVE TABLES
# =========================
spark.sql(f"DROP TABLE IF EXISTS {ACTIVE_USER_TABLE}")
spark.sql(f"DROP TABLE IF EXISTS {OUTPUT_TABLE}")

active_users_df.write.format("delta").mode("overwrite").saveAsTable(ACTIVE_USER_TABLE)
model_df.write.format("delta").mode("overwrite").saveAsTable(OUTPUT_TABLE)

print(f"Saved active users to: {ACTIVE_USER_TABLE}")
print(f"Saved GraphGRU-style weekly base table to: {OUTPUT_TABLE}")
spark.table(OUTPUT_TABLE).printSchema()


StatementMeta(, b736590c-1d4c-4ea5-bc3c-b6ba0c1178b5, 30, Finished, Available, Finished, False)

Saved active users to: dbo.tbl_graphgrued_active_users
Saved GraphGRU-style weekly base table to: dbo.tbl_graphgrued_weekly_active_base
root
 |-- user_id: string (nullable = true)
 |-- course_id: string (nullable = true)
 |-- enroll_time: timestamp (nullable = true)
 |-- gender: string (nullable = true)
 |-- year_of_birth: long (nullable = true)
 |-- school_id: string (nullable = true)
 |-- num_courses_enrolled: double (nullable = true)
 |-- course_name: string (nullable = true)
 |-- about: string (nullable = true)
 |-- prerequisites: string (nullable = true)
 |-- num_fields: double (nullable = true)
 |-- num_resources: double (nullable = true)
 |-- num_active_enrollments: double (nullable = true)
 |-- course_total_videos: double (nullable = true)
 |-- course_max_chapter: double (nullable = true)
 |-- video_watch_time_week1: double (nullable = true)
 |-- video_watch_time_week2: double (nullable = true)
 |-- video_watch_time_week3: double (nullable = true)
 |-- video_watch_time_week4: d